# Sharpening metacells

See `INSTALL.md` for installing what this needs. Nothing here installs anything: if the cell below
fails, the environment is not set up, and the notebook says so rather than working around it.

In [1]:
import os

import numpy as np

import dafpy as dp
import metacellspy as mc
import somegraphspy as sg

# What the packages say while they work. `mcs_results` is the summary of each computation - the mean
# cells in a metacell, the number of blocks - which is worth reading and is a line or two apiece.
# The other groups narrate every call and every loop, which is for working on the packages rather
# than for using them.
#
# Set here rather than left to the environment, so that this notebook says the same thing wherever
# it runs, and so that reading it shows what running it shows.
os.environ["JULIA_DEBUG"] = "mcs_results"

print("dafpy", dp.__version__)
print("somegraphspy", sg.__version__)
print("metacellspy", mc.__version__)

# Read from Julia at import, so printing it means Python reached Julia rather than merely that the
# Python packages are installed.
print("regularization", mc.GENE_FRACTION_REGULARIZATION_FOR_CELLS)

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


[ Info: Will cache ispath data forever
[ Info: Old linux kernel, will pre-populate mmap into RAM disk: Linux version 4.18.0-553.30.1.el8_10.x86_64 (mockbuild@x64-builder01.almalinux.org) (gcc version 8.5.0 20210514 (Red Hat 8.5.0-22) (GCC)) #1 SMP Tue Nov 26 02:30:26 EST 2024


dafpy 0.3.0
somegraphspy 0.2.0
metacellspy 0.1.0
regularization 0.0001


## Importing the cells


In [2]:
# What to take out of the `AnnData`, and under what name. Anything not named here is copied as it is,
# after the importer's own renaming: a `something_cell` or `something_gene` mask arrives as
# `is_something`, and a `something_umis` as `something_UMIs`. Naming a property here overrides that
# for it alone, so the rest of the import is unaffected.
COPY_DATA = {
    # The type of each cell. **Specify this whenever the data has a type per cell**: the type axis is
    # built from a vector called `type`, and the column holding it is rarely called that. Leave it
    # out and everything still runs, with no types and uncolored graphs.
    ("cell", "cell_type"): ("type", None),
    # This dataset has a column of its own called `type` - the platform each cell was measured on,
    # which is not a cell type at all. Left alone it would collide with the line above.
    ("cell", "type"): ("platform", None),
    #
    # The batch, the plate it was on, and the run it was sequenced in. These become axes of their
    # own further down, so they are given the names those axes will have. Three other columns hold
    # the same batch identifier - `batch_set_id` is identical to it, `plate` is it with 1212 cells
    # saying the literal string `NA`, and `Plate` is it with the 10x cells left blank - so they are
    # dropped rather than imported and then explained.
    ("cell", "amp_batch_id"): ("batch", None),
    ("cell", "batch_set_id"): None,
    ("cell", "Plate"): None,
    ("cell", "plate"): None,
    ("cell", "Plate.."): ("plate", None),
    ("cell", "seq_batch_id"): ("sequencing_run", None),
    #
    # The wet lab's record of each batch, plate and run. These are spelled as they were typed into a
    # spreadsheet, dots, capitals, typos and all, and are about to become properties of those axes
    # where they will be read rather than merely stored.
    ("cell", "Comment"): ("comment", None),
    ("cell", "Conc...ng.ul."): ("concentration_ng_per_ul", None),
    ("cell", "Evarage.size..bp."): ("average_size_bp", None),
    ("cell", "External.Index"): ("external_index", None),
    ("cell", "Internal.Index"): ("internal_index", None),
    ("cell", "QC1"): ("qc1", None),
    ("cell", "QC2"): ("qc2", None),
    ("cell", "delta_CT"): ("delta_ct", None),
    ("cell", "Libprep.Cycles"): ("libprep_cycles", None),
    ("cell", "Owner"): ("owner", None),
    ("cell", "Plate.Date"): ("plate_date", None),
    ("cell", "Production.Date"): ("production_date", None),
    ("cell", "Sort.Date"): ("sort_date", None),
    ("cell", "Last.sequensing.date"): ("last_sequencing_date", None),
    ("cell", "Sequencing.Dates"): ("sequencing_dates", None),
    ("cell", "Experiment"): ("experiment", None),
    # Not a genotype: the values are free text describing the sample the batch was made from - the
    # strain, the stage, which embryos, whether it is placenta - and only some of them are strains.
    # It is constant per batch and per plate, and *not* per embryo, which is the giveaway.
    ("cell", "Genotype"): ("description", None),
    #
    # Columns which hold one value, or none at all: `Ref` is `mm10` for every cell that has it,
    # `Empty.Wells` is one list of wells repeated, `X.1` is the string `NA` for all 110,746 cells,
    # `X` is blank for most of them, and `not_na` is true throughout. None of them distinguishes
    # anything, so none of them is worth carrying. Nor does `cell`, which repeats the cell's own name
    # for 64,675 of them and says `NA` for the other 46,071.
    #
    # `X` is why a key names the axis and not just the property: the UMIs matrix is called `X` as
    # well, and naming that one would be `("cell", "gene", "X")`.
    ("cell", "cell"): None,
    ("cell", "Ref"): None,
    ("cell", "Empty.Wells"): None,
    ("cell", "X"): None,
    ("cell", "X.1"): None,
    ("cell", "not_na"): None,
}

cells = dp.files_daf("dafs/cells", "w", name="cells")

# Which metacell each cell belongs to is not imported - it is one analysis of these cells rather than a fact about
# them, and every sharpening round produces another - but the importer has the file open, so it hands it back rather
# than making the next cell read it again. That cell is where it goes, into the repository of the round it answers.
base_metacell_per_cell = mc.import_cells_h5ad(cells, cells_h5ad="input/assigned_cells.h5ad", copy_data=COPY_DATA)

# The total UMIs of a cell look like data and are not: they are the sum of the UMIs already imported. An `h5ad` which
# happens to carry them is imported with them and is left alone; this one does not, so they are computed here, once,
# rather than by whichever computation needs them first.
if not cells.has_vector("cell", "total_UMIs"):
    mc.compute_vector_of_total_UMIs_per_cell(cells)

print(cells.description())

Sum 100%|████████████████████████████████████████████████| Time: 0:00:03


┌ Debug: Mean (included) UMIs per cell: 7862.727421306413
└ @ Metacells.AnalyzeCells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_cells.jl:56


name: cells
type: FilesDaf
path: /net/mraid20/ifs/wisdom/tanay_lab/data/users/obk/src/metacells-sharpening-vignette/dafs/cells
mode: w
axes:
  cell: 110746 entries
  gene: 28183 entries
vectors:
  cell:
    age_group: 110,746 x Float64 (Dense)
    age_group_emb: 110,746 x Float64 (Dense)
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    average_size_bp: 110,746 x Str (Dense)
    batch: 110,746 x Str (Dense)
    comment: 110,746 x Str (Dense)
    concentration_ng_per_ul: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    delta_ct: 110,746 x Str (Dense)
    description: 110,746 x Str (Dense)
    developmental_time: 110,746 x Float64 (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x Str (Dense)
    embryo_with_placenta_information: 110,746 x Str (Dense)
    excluded_UMIs: 110,746 x UInt32 (Dense)
    experiment: 110,746 x Str (Dense)
    external_index: 110,74

## Cleaning the data


In [3]:
# How this data spells "there is no value here", which is not one way but several, sometimes several
# in the same property: `embryo` says both `NA` and nothing at all. Nothing infers these - a type
# genuinely called `NA` is possible - so each is named, and a property named here whose data happens
# to be clean is simply left alone.
#
# This has to happen before any axis is built, since building one asks of each cell whether it has a
# value: a property still saying `NA` would put `NA` on the axis, sitting among the real entries.
EMPTY_VALUES = {
    "embryo": ("NA",),
    "type": ("Outliers", "Doublet"),
    "projected_type": ("(Missing)",),
    "coordinates": ("NA",),
    "source": ("NA",),
}

for property_name, empty_values in EMPTY_VALUES.items():
    dp.unify_empty_vector_values(cells, axis="cell", property=property_name, empty_values=empty_values)

# Numbers which arrived as text, because a few of their entries say `NA` and one `NA` makes a whole
# column of measurements a column of strings. Converting and unifying is one step, not two: what
# `23.5` should become is obvious, and what `NA` should become is only obvious once we are told that
# it means nothing. A number which is neither is an error rather than a silent `NaN`.
AS_NUMBERS = {
    "qc1": ("NA", np.float32),
    "qc2": ("NA", np.float32),
    "delta_ct": ("NA", np.float32),
    "concentration_ng_per_ul": ("NA", np.float32),
    # 1..32, so `0` is free to mean "none" - the convention `Daf` already uses for module indices.
    # The cells with no plate are the ones sequenced by 10x, which has no plates.
    "internal_index": ("", np.uint32),
}

for property_name, (empty_values, dtype) in AS_NUMBERS.items():
    dp.unify_empty_vector_values(
        cells, axis="cell", property=property_name, empty_values=empty_values, dtype=dtype
    )

# A sentinel which is not obviously one: the smallest 32 bit integer, which survived a cast to float
# and so is an ordinary number as far as anything reading it is concerned. Left alone, the mean of
# this property is wrong by a couple of billion rather than visibly absent.
dp.unify_empty_vector_values(
    cells, axis="cell", property="transcriptional_rank", empty_values=np.float64(-2147483648.0)
)

## Reconstructing the axes


In [4]:
# The types, and the color of each, which is what makes the graphs readable. The file decides which
# types there are and in what order they are listed - usually a meaningful order rather than an
# alphabetical one. It may name a type no cell has; a type of some cell which it does not name is an
# error, in the file or in the data. Skip this and everything still runs, uncolored.
mc.import_type_colors_csv(cells, type_colors_csv="input/type_colors.csv")

# `AnnData` has two axes, so everything else it knows is flattened onto the cells: which batch a cell
# came from, and with it every fact about that batch, repeated across its cells. Reconstructing an
# axis puts each fact where it belongs - one value per batch rather than 110,746 copies of it - and
# says so in the structure rather than in a naming convention.
#
# What is per batch, and what is merely constant within a batch by accident, is decided by looking:
# a property whose value differs between two cells of a batch is left alone. That is convenient and
# slightly dangerous, since a property which happens to be uniform is moved as readily as one which
# is uniform for a reason. These are the pipeline's own, which belong to the cells whatever their
# values happen to look like here - `is_excluded` is false for every cell of this data set, which
# says nothing about where it belongs.
KEEP_PER_CELL = {"is_excluded", "is_properly_sampled", "is_rare", "rare_gene_module", "spike_count"}

for axis in ("batch", "embryo"):
    dp.reconstruct_axis(cells, existing_axis="cell", implicit_axis=axis, skipped_properties=KEEP_PER_CELL)

# A batch was on a plate and was sequenced in a run, so those are properties of the batch now, and
# each is an axis of its own with the batch's facts divided again between them. The wet lab's record
# lands where it is read: the plate's owner and dates on the plate, the batch's concentration and QC
# on the batch, the sequencing dates on the run.
#
# The coarser axis goes first. Each plate was sequenced in one run, so a fact about a run is also
# constant within each of its plates, and reconstructing the plate first would take the run's dates
# with it - leaving the run with nothing. The reverse cannot happen: a run holds many plates, so a
# plate's own owner and dates are not constant within it.
for axis in ("sequencing_run", "plate"):
    dp.reconstruct_axis(cells, existing_axis="batch", implicit_axis=axis, skipped_properties=KEEP_PER_CELL)

# Each plate belongs to one sequencing run, but nothing has said so where a plate can be asked. It
# cannot be reconstructed: the cells sequenced by 10x have a run and no plate at all, so moving the
# run onto the plate would discard theirs. Connecting says it while leaving the batch's own run
# alone, and fails if any plate's batches disagree about which run they were in.
dp.connect_axes(cells, base_axis="batch", from_axis="plate", to_axis="sequencing_run")

print(cells.description())

name: cells
type: FilesDaf
path: /net/mraid20/ifs/wisdom/tanay_lab/data/users/obk/src/metacells-sharpening-vignette/dafs/cells
mode: w
axes:
  batch: 416 entries
  cell: 110746 entries
  embryo: 385 entries
  gene: 28183 entries
  plate: 246 entries
  sequencing_run: 195 entries
  type: 44 entries
vectors:
  batch:
    average_size_bp: 416 x Str (Dense)
    comment: 416 x Str (Dense)
    concentration_ng_per_ul: 416 x Float32 (Dense)
    delta_ct: 416 x Float32 (Dense)
    external_index: 416 x Str (Dense)
    internal_index: 416 x UInt32 (Dense)
    plate: 416 x Str (Dense)
    qc1: 416 x Float32 (Dense)
    qc2: 416 x Float32 (Dense)
    sequencing_run: 416 x Str (Dense)
  cell:
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    batch: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x Str (Dense)
    embryo_with_place

## Building the base metacells


In [5]:
# The metacells we start from, in a repository of their own resting on the cells. Everything computed
# from here on lives in such a repository, so that one set of cells can carry several analyses of
# them without copying a single UMI.
base_metacells = dp.complete_chain(
    base_daf=cells,
    new_daf=dp.files_daf("dafs/metacells.base", "w", name="metacells.base"),
    name="metacells.base",
)

# Which metacell each cell belongs to, which came with the `h5ad` rather than being computed. It
# goes into this repository rather than into the cells: it is one analysis of them, not a fact about
# them, and every sharpening round writes an assignment of its own.
#
# What does go into the cells is which of them this assignment left out, since that is the one thing
# about it the later rounds still need and cannot work out for themselves: a cell a round ejects also
# has no metacell, and the round after it may place that cell, but a cell the metacells we start from
# never placed stays out for good.
#
# `Outliers` is how this data spells "no metacell", the same way `clean_data` was told how it spells
# the rest of them. Left unsaid, `Outliers` would become a metacell of its own on the axis below.
mc.import_base_metacells(
    cells_daf=cells,
    metacells_daf=base_metacells,
    metacell_per_cell=base_metacell_per_cell,
    empty_metacells=("Outliers",),
    overwrite=True,
)

# The metacells themselves, named by the values of that property. `reconstruct_axis` would also move
# to the new axis every per-cell property which happens to be constant per metacell; here it is asked
# for the axis alone, since anything per metacell is about to be computed rather than inherited.
dp.reconstruct_axis(
    base_metacells, existing_axis="cell", implicit_axis="metacell", implicit_properties=set()
)

# What the cells say about their metacells: how many cells each has, their UMIs, and the fraction of
# each gene in each of them. Then the marker genes - the ones which distinguish between metacells.
#
# Neither depends on the gene masks, which is why they are here rather than in each analysis: this
# repository is shared by all of them.
mc.prepare_metacells(base_metacells)
mc.prepare_markers(base_metacells)

print(base_metacells.description())

┌ Debug: Base outlier cells: 39 out of 110746
└ @ Metacells.AnalyzeCells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_cells.jl:98
GroupBy(Mode) 100%|██████████████████████████████████████| Time: 0:00:00


GroupByColumns(Sum) 100%|████████████████████████████████| Time: 0:00:09
Sum 100%|████████████████████████████████████████████████| Time: 0:00:00
┌ Debug: Mean UMIs in metacell: 361093.6524263791
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_metacells.jl:145
GroupBy(Count) 100%|█████████████████████████████████████| Time: 0:00:00


┌ Debug: Mean cells in metacell: 45.91746163417669
└ @ Metacells.AnalyzeMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_metacells.jl:92


log_linear_fraction_per_gene_per_metacell 100%|██████████| Time: 0:00:01


Max 100%|████████████████████████████████████████████████| Time: 0:00:00
Min 100%|████████████████████████████████████████████████| Time: 0:00:00
Max 100%|████████████████████████████████████████████████| Time: 0:00:00


┌ Debug: Marker genes: 8349
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_genes.jl:100
Median 100%|█████████████████████████████████████████████| Time: 0:00:00
abs_fold_per_metacell_per_marker 100%|███████████████████| Time: 0:00:00


rank_per_variable_per_observation 100%|██████████████████| Time: 0:00:01
min_rank_and_maximal_score_per_variable 100%|████████████| Time: 0:00:00


name: metacells.base#2
type: Write Chain
chain:
- FilesDaf cells
- FilesDaf metacells.base
scalars:
  base_daf_repository: "cells"
axes:
  batch: 416 entries
  cell: 110746 entries
  embryo: 385 entries
  gene: 28183 entries
  metacell: 2411 entries
  plate: 246 entries
  sequencing_run: 195 entries
  type: 44 entries
vectors:
  batch:
    average_size_bp: 416 x Str (Dense)
    comment: 416 x Str (Dense)
    concentration_ng_per_ul: 416 x Float32 (Dense)
    delta_ct: 416 x Float32 (Dense)
    external_index: 416 x Str (Dense)
    internal_index: 416 x UInt32 (Dense)
    plate: 416 x Str (Dense)
    qc1: 416 x Float32 (Dense)
    qc2: 416 x Float32 (Dense)
    sequencing_run: 416 x Str (Dense)
  cell:
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    batch: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x Str (Dense)


## Choosing the gene masks

In [6]:
# Which genes are what, which is the one thing an iteration of this analysis is free to disagree about. Everything
# computed from here on depends on these four masks, and on nothing else which varies, so a repository of them is what
# an iteration *is*: change them, and every result below changes with them.
masks = dp.complete_chain(
    base_daf=cells,
    new_daf=dp.files_daf("dafs/masks.I0", "w", name="masks.I0"),
    name="masks.I0",
)

n_genes = masks.axis_length("gene")

# Genes which may not be used to predict the others, whatever the data says about them. There is no such list to start
# with: this first iteration takes the data as it comes, so that the second has something to be better than.
masks.set_vector("gene", "is_forbidden", np.zeros(n_genes, dtype=bool))

# `is_lateral` - the genes which are not to drive the metacells - is left exactly as the `h5ad` carried it, for the
# same reason. Patching it is the other half of what a later iteration does.

# Which genes regulate others, and which are transcription factors, fetched from Gmara rather than decided here. These
# describe the species and not this experiment, so they do not vary between iterations; they live here only because
# `is_regulator` is one of the four masks, and keeping the set together is what makes an iteration one thing.
mc.fetch_gmara_vector_of_is_regulator_per_gene(masks, species="mouse")
mc.fetch_gmara_vector_of_is_transcription_factor_per_gene(masks, species="mouse")

print(masks.description())

┌ Debug: Regulators: 271
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_genes.jl:524
┌ Debug: Transcription factors: 1780
└ @ Metacells.AnalyzeGenes ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_genes.jl:491


name: masks.I0#2
type: Write Chain
chain:
- FilesDaf cells
- FilesDaf masks.I0
scalars:
  base_daf_repository: "cells"
axes:
  batch: 416 entries
  cell: 110746 entries
  embryo: 385 entries
  gene: 28183 entries
  plate: 246 entries
  sequencing_run: 195 entries
  type: 44 entries
vectors:
  batch:
    average_size_bp: 416 x Str (Dense)
    comment: 416 x Str (Dense)
    concentration_ng_per_ul: 416 x Float32 (Dense)
    delta_ct: 416 x Float32 (Dense)
    external_index: 416 x Str (Dense)
    internal_index: 416 x UInt32 (Dense)
    plate: 416 x Str (Dense)
    qc1: 416 x Float32 (Dense)
    qc2: 416 x Float32 (Dense)
    sequencing_run: 416 x Str (Dense)
  cell:
    alexa_fluor_488_a: 110,746 x Float64 (Dense)
    apc_a: 110,746 x Float64 (Dense)
    apc_cy7_a: 110,746 x Float64 (Dense)
    batch: 110,746 x Str (Dense)
    coordinates: 110,746 x Str (Dense)
    dissolved: 110,746 x Bool (Sparse 2 (<1%) [UInt32])
    embryo: 110,746 x Str (Dense)
    embryo_with_placenta_information:

## Analyzing the metacells

In [7]:
# The first round of the first iteration. It rests on two repositories: the metacells the cells were aggregated into,
# which every iteration shares, and this iteration's gene masks. Both rest in turn on the same cells, so the cells are
# reached through either arm and are used once.
#
# The name says which iteration and which round: `I0` is this set of masks and `R0` is the metacells before any
# sharpening. Each round of sharpening writes an `R1`, an `R2` and so on beside this one, and each iteration starts
# again at its own `R0`.
metacells = dp.complete_chain(
    base_daf=[base_metacells, masks],
    new_daf=dp.files_daf("dafs/metacells.I0.R0", "w", name="metacells.I0.R0"),
    name="metacells.I0.R0",
)

# Everything these metacells say about the manifold: which genes are skeleton, how far the metacells are from each
# other and how they lay out, the blocks they fall into with their neighborhoods and environments, and the gene modules
# of each block. This is what sharpening reads, so it is also what has to be recomputed after each round of it.
#
# `module_status` records why each gene ended up in the module it did, which is worth having while reading a result.
mc.analyze_metacells(metacells, module_status=True)

print(metacells.description())

┌ Debug: Skeletons: 28183 [ Alx1, Ankrd1, Arg1, Arid3b, Arid5b, Bach1, Bcl11a, Cdx1, Cdx2, Cdx4, Cebpb, Cebpz, Creb1, Dlx3, Dlx5, Dlx6, Dppa2, Dppa3, Dppa4, E2f4, Ebf2, Egr1, Egr2, Elf1, Elf2, Elk3, En1, Eomes, Epas1, Erg, Ets1, Ets2, Etv2, Etv5, Evx1, Fli1, Fos, Fosl2, Foxa1, Foxa2, Foxc1, Foxc2, Foxd1, Foxd3, Foxf1, Foxj1, Foxo1, Foxo3, Foxo4, Foxq1, Gata1, Gata2, Gata3, Gata4, Gata5, Gata6, Gbx2, Gfi1b, Glis1, Gsc, Gtf3a, Hand1, Hand2, Hes1, Hes3, Hes5, Hes6, Hes7, Hesx1, Hhex, Hlx, Hnf4a, Hopx, Hoxa1, Hoxa10, Hoxa3, Hoxa5, Hoxa7, Hoxa9, Hoxb1, Hoxb4, Hoxb6, Hoxb9, Hoxc5, Hoxc8, Hoxc9, Hoxd3, Hoxd9, Id1, Ikzf1, Ikzf2, Ilf2, Irf1, Irx2, Irx3, Irx5, Isl1, Jund, Klf1, Klf13, Klf2, Klf3, Klf4, Klf5, Klf6, Klf7, Klf8, Klf9, Lef1, Lhx1, Lhx2, Lmo2, Lmx1a, Lmx1b, Lyl1, Maf, Mafb, Mafg, Meis2, Meox1, Mesp1, Mixl1, Msx1, Msx2, Mxi1, Myb, Myc, Nanog, Nfatc3, Nfe2, Nfkb1, Nfyb, Nkx1-2, Nkx2-5, Noto, Nr2f1, Nr2f2, Olig1, Otx1, Otx2, Pax2, Pax3, Pax6, Pbx1, Pitx1, Pou3f1, Prdm1, Prrx2, Rara, Rar

high_log_linear_fraction_per_skeleton_per_metacell 100%|█| Time: 0:00:00


max_skeleton_fold_distance_between_metacells 100%|███████| Time: 0:00:03


euclidean_skeleton_fold_distance_between_metacells 100%|█| Time: 0:00:01
┌ Debug: Blocks: 408
└ @ Metacells.ComputeBlocks ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/compute_blocks.jl:91
GroupByColumns(Mean) 100%|███████████████████████████████| Time: 0:00:00
GroupByColumns(Mean) 100%|███████████████████████████████| Time: 0:00:00


GroupBy(Count) 100%|█████████████████████████████████████| Time: 0:00:00
┌ Debug: Mean metacells in block: 5.909313725490196
└ @ Metacells.AnalyzeBlocks ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_blocks.jl:156
GroupBy(Sum) 100%|███████████████████████████████████████| Time: 0:00:00
┌ Debug: Mean cells in block: 271.3406862745098
└ @ Metacells.AnalyzeBlocks ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_blocks.jl:183


...

most_correlated_pertinent_neighborhood_markers_per_gene_per_block 100% Time: 0:00:25


compute_blocks_modules 100%|█████████████████████████████| Time: 0:00:14
Sum 100%|████████████████████████████████████████████████| Time: 0:00:00
┌ Debug: Mean found modules per block: 18.279411764705884
└ @ Metacells.AnalyzeModules ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_modules.jl:79


n_genes_per_module_per_block 100%|███████████████████████| Time: 0:00:00
┌ Debug: Mean genes per found module: 47.54237060874229
└ @ Metacells.AnalyzeModules ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_modules.jl:122


stats_of_linear_fraction_in_environment_cells_per_module_per_block 100% Time: 0:00:02
cells_dispersion_per_metacell_per_module 100%|███████████| Time: 0:00:00
┌ Debug: Mean max cells dispersion per metacell per found module: 2.1908534
└ @ Metacells.AnalyzeModules ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_modules.jl:953


name: metacells.I0.R0#2
type: Write Chain
chain:
- FilesDaf cells
- FilesDaf metacells.base
- FilesDaf masks.I0
- FilesDaf metacells.I0.R0
scalars:
  base_daf_repository: "["metacells.base..." (29)
axes:
  batch: 416 entries
  block: 408 entries
  cell: 110746 entries
  embryo: 385 entries
  gene: 28183 entries
  metacell: 2411 entries
  module: 197 entries
  plate: 246 entries
  sequencing_run: 195 entries
  type: 44 entries
vectors:
  batch:
    average_size_bp: 416 x Str (Dense)
    comment: 416 x Str (Dense)
    concentration_ng_per_ul: 416 x Float32 (Dense)
    delta_ct: 416 x Float32 (Dense)
    external_index: 416 x Str (Dense)
    internal_index: 416 x UInt32 (Dense)
    plate: 416 x Str (Dense)
    qc1: 416 x Float32 (Dense)
    qc2: 416 x Float32 (Dense)
    sequencing_run: 416 x Str (Dense)
  block:
    n_cells: 408 x UInt32 (Dense)
    n_environment_cells: 408 x UInt32 (Dense)
    n_environment_metacells: 408 x UInt32 (Dense)
    n_metacells: 408 x UInt32 (Dense)
    n_modu

## Sharpening the metacells

In [8]:
# Sharpening, which is the point of all of the above. A round re-groups the cells using what the round before it worked
# out about the manifold, and then works the manifold out again from the groups it arrived at - so each round is the
# same five calls, reading the round before it and writing a repository of its own.
#
# Two rounds here. Each is cheap to add and none of them is the last word, so how many to run is a number rather than a
# decision: raise it and the rounds below it are untouched, since each already ran and wrote what it wrote.
SHARPENING_ROUNDS = 2

for sharpening_round in range(1, SHARPENING_ROUNDS + 1):
    # The round's own repository, resting on the gene masks and through them on the cells. Not on the round before it:
    # what that round grouped the cells into is what this one is about to disagree with, and a repository can only hold
    # one set of metacells.
    name = f"metacells.I0.R{sharpening_round}"
    sharpened = dp.complete_chain(
        base_daf=masks,
        new_daf=dp.files_daf(f"dafs/{name}", "w", name=name),
        name=name,
    )

    # Which cells belong together, decided again - by clustering each neighborhood on the gene modules the previous
    # round found there. Cells which fit nothing are ejected, and are free to be placed again by the round after this.
    #
    # Each round advances the letter its metacells and its blocks are named with, so a name says which round it came
    # from wherever it turns up. The metacells we started from are the `M` the `h5ad` named them and their blocks the
    # `B` of the analysis above - round zero, in effect - so each round here is that many letters further on.
    mc.sharpen_metacells(
        sharp_daf=sharpened,
        base_daf=metacells,
        prefix=chr(ord("M") + sharpening_round),
        sharpening_round=sharpening_round,
    )

    # The same three steps the metacells we started from went through, now that this round has said which cells are
    # which metacell: aggregate the cells into them, find the genes which tell them apart, and work out what they say
    # about the manifold. That last is what the next round reads.
    mc.prepare_metacells(sharpened)
    mc.prepare_markers(sharpened)
    mc.analyze_metacells(
        sharpened,
        prefix=chr(ord("B") + sharpening_round),
        prev_daf=metacells,
        module_status=True,
    )

    metacells = sharpened

print(metacells.description())

target_min_cells_per_block 100%|█████████████████████████| Time: 0:00:00
Max 100%|████████████████████████████████████████████████| Time: 0:00:00
┌ Debug: Adding metacells: 172 to base: 2411 due to cells dispersion
└ @ Metacells.SharpenMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/sharpen_metacells.jl:456


preferred_block_index_per_cell_per_block 100%|███████████| Time: 0:00:04


preferred_block_index_of_cells 100%|█████████████████████| Time: 0:00:00
┌ Debug: Cells: 110746 Stationary: 86457 (78%) Restless: 18506 (17%) Migrated: 5744 (5%)
└ @ Metacells.SharpenMetacells ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/sharpen_metacells.jl:969


clustering_options 100%|█████████████████████████████████| Time: 0:00:13
precompute_walkable_indirection 100%|████████████████████| Time: 0:00:00


UMIs_cache_fill 100%|████████████████████████████████████| Time: 0:00:00


...

is_correlated_with_skeleton_in_environment_per_gene_per_block 100% Time: 0:00:08
┌ Debug: Mean markers correlated with skeleton in environment: 3416.586118251928
└ @ Metacells.AnalyzeBlocks ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_blocks.jl:1857


most_correlated_pertinent_neighborhood_markers_per_gene_per_block 100% Time: 0:00:13


compute_blocks_modules 100%|█████████████████████████████| Time: 0:00:07
Sum 100%|████████████████████████████████████████████████| Time: 0:00:00
┌ Debug: Mean found modules per block: 21.34190231362468
└ @ Metacells.AnalyzeModules ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_modules.jl:79
n_genes_per_module_per_block 100%|███████████████████████| Time: 0:00:00
┌ Debug: Mean genes per found module: 79.19537460852807
└ @ Metacells.AnalyzeModules ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_modules.jl:122


stats_of_linear_fraction_in_environment_cells_per_module_per_block 100% Time: 0:00:01
cells_dispersion_per_metacell_per_module 100%|███████████| Time: 0:00:00
┌ Debug: Mean max cells dispersion per metacell per found module: 2.0383976
└ @ Metacells.AnalyzeModules ~/anaconda3/envs/metacells-sharpening/share/julia/packages/Metacells/KcW9X/src/analyze_modules.jl:953


name: metacells.I0.R2#2
type: Write Chain
chain:
- FilesDaf cells
- FilesDaf masks.I0
- FilesDaf metacells.I0.R2
scalars:
  base_daf_repository: "masks.I0"
axes:
  batch: 416 entries
  block: 389 entries
  cell: 110746 entries
  embryo: 385 entries
  gene: 28183 entries
  metacell: 1576 entries
  module: 201 entries
  plate: 246 entries
  sequencing_run: 195 entries
  type: 44 entries
vectors:
  batch:
    average_size_bp: 416 x Str (Dense)
    comment: 416 x Str (Dense)
    concentration_ng_per_ul: 416 x Float32 (Dense)
    delta_ct: 416 x Float32 (Dense)
    external_index: 416 x Str (Dense)
    internal_index: 416 x UInt32 (Dense)
    plate: 416 x Str (Dense)
    qc1: 416 x Float32 (Dense)
    qc2: 416 x Float32 (Dense)
    sequencing_run: 416 x Str (Dense)
  block:
    n_cells: 389 x UInt32 (Dense)
    n_environment_cells: 389 x UInt32 (Dense)
    n_environment_metacells: 389 x UInt32 (Dense)
    n_metacells: 389 x UInt32 (Dense)
    n_modules: 389 x UInt32 (Dense)
    n_neighborho